# Этап 3: Prompt-based классификация (Zero / One / Few-shot)

Классификация писем через промпты с генеративными LLM.

**Модели:** saiga_8b, t_lite_8b, vikhr_12b, qwen_14b, qwen_32b

**Эксперименты:**
| Ключ | K | Описания | max_chars | Описание |
|------|---|----------|-----------|----------|
| 0    | 0 | да       | 500       | zero-shot |
| 1    | 1 | да       | 500       | one-shot |
| 3    | 3 | да       | 500       | few-shot K=3 с описаниями |
| 5    | 5 | да       | 500       | few-shot K=5 с описаниями (может не влезть) |
| 3nd  | 3 | нет      | 500       | few-shot K=3 без описаний |
| 5s   | 5 | да       | 300       | few-shot K=5, короткие примеры |
| 5snd | 5 | нет      | 300       | few-shot K=5, короткие примеры, без описаний |

___
## Подготовка окружения (Запуск в Colab)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys
from pathlib import Path

# Корень проекта на Google Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/VKR/code")
sys.path.insert(0, str(PROJECT_ROOT))

## Подготовка окружения (Локальный запуск)

In [ ]:
# import sys
# from pathlib import Path
# PROJECT_ROOT = Path(".").resolve().parent
# sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# Ставим зависимости из requirements.txt
!pip install -q -r {PROJECT_ROOT}/requirements.txt

## Импорты и конфигурация

In [ ]:
import json
import gc
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_loader import load_dataset, load_test_set, TEXT_COL, LABEL_COL
from src.classification.few_shot_examples import (
    prepare_few_shot_examples, load_few_shot_examples,
)
from src.classification.prompt_classifier import (
    load_prompt_config, load_model, unload_model,
    build_prompt, classify_dataset, load_class_descriptions,
    apply_chat_template,
)
from src.classification.evaluate import evaluate_prompt_classification

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

DATA_DIR = PROJECT_ROOT / "Data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Загрузка конфига prompt_classification
cfg = load_prompt_config()
gen_params = cfg["generation_params"]
prompt_models = cfg["prompt_models"]

print("Модели:")
for key, m in prompt_models.items():
    print(f"  {key}: {m['model_name']} (ctx={m['max_context']}, vram~{m['vram_gb']}GB)")

In [ ]:
# Загрузка данных
df_train = load_dataset(stage=0)
df_test = load_test_set()

labels = sorted(df_train[LABEL_COL].unique().tolist())

print(f"\nТест: {len(df_test)} примеров, Классов: {len(labels)}")

___
## Шаг 1: Генерация описаний классов

Используем Qwen-32B-AWQ для генерации описаний всех 36 классов.
Описания нужны для промптов, чтобы модель лучше понимала каждый класс.

In [ ]:
desc_path = DATA_DIR / "class_descriptions.json"

if desc_path.exists():
    descriptions = load_class_descriptions(desc_path)
    print(f"Описания загружены из кэша ({len(descriptions)} классов)")
else:
    from src.augmentation.stage1_llm_generate import generate_class_context
    from src.augmentation.llm_utils import load_llm

    config_path = str(PROJECT_ROOT / "config_models" / "aug_configs" / "model_vllm_32b.json")
    llm, sampling_params, system_prompt = load_llm(config_path)

    descriptions = {}
    for cls in sorted(labels):
        examples = df_train[df_train[LABEL_COL] == cls][TEXT_COL].tolist()[:5]
        desc = generate_class_context(cls, examples, llm, sampling_params, system_prompt)
        descriptions[cls] = desc

    with open(desc_path, "w", encoding="utf-8") as f:
        json.dump(descriptions, f, ensure_ascii=False, indent=2)

    print(f"Сгенерировано и сохранено {len(descriptions)} описаний")

    del llm, sampling_params, system_prompt
    gc.collect()
    torch.cuda.empty_cache()

# Пример
first_cls = sorted(descriptions.keys())[0]
print(f"\n{first_cls}:\n{descriptions[first_cls]}")

___
## Шаг 2: Подготовка few-shot примеров

Для K=1, 3, 5 с учётом групп:
- **A** (9 кл., 50+) и **B** (15 кл., 15-49): по K примеров
- **C** (12 кл., <15): все доступные примеры из train

In [ ]:
fs_path = DATA_DIR / "few_shot_examples.json"

if fs_path.exists():
    fs_data = load_few_shot_examples(fs_path)
    print("Few-shot примеры загружены из кэша")
else:
    fs_data = prepare_few_shot_examples(k_values=[1, 3, 5])
    print("Few-shot примеры сгенерированы и сохранены")

groups = fs_data["groups"]

for k in ["1", "3", "5"]:
    total = sum(len(v) for v in fs_data["examples"][k].values())
    print(f"K={k}: {total} примеров")

groups_counts = {g: sum(1 for v in groups.values() if v == g) for g in "ABC"}
print(f"Группы: A={groups_counts['A']}, B={groups_counts['B']}, C={groups_counts['C']}")

___
## Шаг 3: Запуск экспериментов

Матрица: 5 моделей x 7 режимов.

| Модель | 0 | 1 | 3 | 3nd | 5s | 5snd |
|--------|---|---|---|-----|----|----- |
| saiga_8b | + | + | - | - | - | - |
| t_lite_8b | + | + | + | + | + | + |
| vikhr_12b | + | + | + | + | + | + |
| qwen_14b | + | + | + | + | + | + |
| qwen_32b | + | + | + | + | + | + |

In [ ]:
# Конфигурация экспериментов
# Каждый эксперимент: (k, mode, no_desc, max_example_chars)
EXPERIMENT_CONFIGS = {
    0:      (0, "zero_shot",  False, 500),
    1:      (1, "one_shot",   False, 500),
    3:      (3, "few_shot",   False, 500),
    5:      (5, "few_shot",   False, 500),
    "3nd":  (3, "few_shot",   True,  500),   # K=3 без описаний
    "5s":   (5, "few_shot",   False, 300),   # K=5 короткие примеры с описаниями
    "5snd": (5, "few_shot",   True,  300),   # K=5 короткие примеры без описаний
}

# Матрица экспериментов: модель -> список ключей из EXPERIMENT_CONFIGS
experiment_matrix = {
    "saiga_8b":     [0, 1],
    "t_lite_8b":    [0, 1, 3, "3nd", "5s", "5snd"],
    "vikhr_12b":    [0, 1, 3, "3nd", "5s", "5snd"],
    "qwen_14b":     [0, 1, 3, "3nd", "5s", "5snd"],
    "qwen_32b":     [0, 1, 3, "3nd", "5s", "5snd"],
}

In [ ]:
# Подхватываем уже посчитанные результаты (если ноутбук перезапущен)
results_path = RESULTS_DIR / "prompt_results.csv"
if results_path.exists():
    df_existing = pd.read_csv(results_path)
    # Совместимость со старым форматом где не было exp_key
    if "exp_key" not in df_existing.columns:
        df_existing["exp_key"] = df_existing["k_shots"].astype(str)
    all_results = df_existing.to_dict("records")
    done = {(r["model"], str(r["exp_key"])) for r in all_results}
    print(f"Загружено {len(all_results)} готовых экспериментов: {done}")
else:
    all_results = []
    done = set()
    print("Предыдущих результатов нет, запуск с нуля")

In [ ]:
def log_raw_responses(df_preds, model_key, exp_key, n_samples=10):
    """Логирует raw_response для отладки extract_prediction."""
    df = df_preds[~df_preds["skipped"]].copy()
    unknowns = df[df["predicted_label"] == "unknown"]
    correct = df[df["predicted_label"] == df["true_label"]]
    wrong = df[
        (df["predicted_label"] != "unknown") &
        (df["predicted_label"] != df["true_label"])
    ]

    print(f"\n{'\u2500'*60}")
    print(f"RAW RESPONSE LOG: {model_key} exp={exp_key}")
    print(f"Correct: {len(correct)} | Wrong: {len(wrong)} | Unknown: {len(unknowns)}")
    print(f"{'\u2500'*60}")

    if len(unknowns) > 0:
        print(f"\n  UNKNOWN ({len(unknowns)} шт, первые {min(n_samples, len(unknowns))}):")
        for _, row in unknowns.head(n_samples).iterrows():
            raw = str(row["raw_response"])[:150]
            print(f"    true: {row['true_label']}")
            print(f"    raw:  {raw}")
            print()

    if len(wrong) > 0:
        print(f"  WRONG ({len(wrong)} шт, первые {min(n_samples, len(wrong))}):")
        for _, row in wrong.head(n_samples).iterrows():
            raw = str(row["raw_response"])[:150]
            print(f"    true: {row['true_label']}")
            print(f"    pred: {row['predicted_label']}")
            print(f"    raw:  {raw}")
            print()

    if len(correct) > 0:
        print(f"  CORRECT (первые {min(3, len(correct))}):")
        for _, row in correct.head(3).iterrows():
            raw = str(row["raw_response"])[:150]
            print(f"    true: {row['true_label']}")
            print(f"    raw:  {raw}")
            print()

    print(f"{'\u2500'*60}")


def run_experiment(model_key, exp_key, model, tokenizer, model_cfg):
    """Запускает один эксперимент: модель x exp_key."""
    max_context = model_cfg["max_context"]

    k_shot, mode, no_desc, max_example_chars = EXPERIMENT_CONFIGS[exp_key]

    if k_shot == 0:
        examples = None
    elif k_shot == 1:
        examples = fs_data["examples"]["1"]
    else:
        examples = fs_data["examples"][str(k_shot)]

    print(f"\n{'='*60}")
    print(f"Эксперимент: {model_key} | exp={exp_key} K={k_shot} mode={mode} no_desc={no_desc} max_chars={max_example_chars}")
    print(f"{'='*60}")

    # Проверка длины промпта на первом примере
    test_prompt = build_prompt(
        df_test.iloc[0][TEXT_COL], labels, descriptions, mode, examples,
        no_desc=no_desc, max_example_chars=max_example_chars,
    )
    test_prompt_formatted = apply_chat_template(tokenizer, test_prompt)
    test_tokens = len(tokenizer.encode(test_prompt_formatted))
    print(f"Длина тестового промпта: {test_tokens} токенов (лимит: {max_context})")

    if test_tokens >= max_context - 100:
        print(f"SKIP: промпт ({test_tokens}) >= лимит ({max_context - 100})")
        return {
            "model": model_key,
            "model_name": model_cfg["model_name"],
            "model_size": model_cfg["vram_gb"],
            "exp_key": str(exp_key),
            "k_shots": k_shot,
            "no_desc": no_desc,
            "max_example_chars": max_example_chars,
            "skipped": True,
            "prompt_tokens": test_tokens,
            "n_test": len(df_test),
        }

    # Классификация
    df_preds = classify_dataset(
        df_test, model, tokenizer, labels, descriptions,
        mode=mode, gen_params=gen_params,
        max_context=max_context, examples=examples,
        fuzzy_cutoff=cfg["extract_prediction"]["fuzzy_cutoff"],
        no_desc=no_desc, max_example_chars=max_example_chars,
    )

    # Оценка
    metrics = evaluate_prompt_classification(df_preds, groups)
    log_raw_responses(df_preds, model_key, exp_key)

    # Сохраняем предсказания
    preds_path = RESULTS_DIR / f"preds_{model_key}_exp{exp_key}.csv"
    df_preds.to_csv(preds_path, index=False)

    return {
        "model": model_key,
        "model_name": model_cfg["model_name"],
        "model_size": model_cfg["vram_gb"],
        "exp_key": str(exp_key),
        "k_shots": k_shot,
        "no_desc": no_desc,
        "max_example_chars": max_example_chars,
        "balanced_accuracy": metrics["balanced_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "unknown_rate": metrics["unknown_rate"],
        "f1_group_A": metrics.get("f1_group_A"),
        "f1_group_B": metrics.get("f1_group_B"),
        "f1_group_C": metrics.get("f1_group_C"),
        "prompt_tokens": test_tokens,
        "skipped": False,
        "n_test": len(df_test),
    }

In [ ]:
# Основной цикл: загружаем модель, прогоняем все эксперименты, выгружаем
for model_key, exp_keys in experiment_matrix.items():
    model_cfg = prompt_models[model_key]

    # Фильтруем только те эксперименты, которые ещё не посчитаны
    exp_todo = [e for e in exp_keys if (model_key, str(e)) not in done]

    if not exp_todo:
        print(f"\n[SKIP] {model_cfg['model_name']} — все эксперименты уже посчитаны")
        continue

    print(f"\n{'#'*60}")
    print(f"# МОДЕЛЬ: {model_cfg['model_name']}")
    print(f"# Эксперименты: {exp_todo}")
    print(f"{'#'*60}")

    try:
        model, tokenizer = load_model(model_cfg)
    except Exception as e:
        print(f"[ERROR] Не удалось загрузить {model_cfg['model_name']}: {e}")
        print("Пропускаю, перехожу к следующей модели...")
        continue

    for exp_key in exp_todo:
        result = run_experiment(model_key, exp_key, model, tokenizer, model_cfg)
        all_results.append(result)
        done.add((model_key, str(exp_key)))

        # Промежуточное сохранение после каждого эксперимента
        pd.DataFrame(all_results).to_csv(results_path, index=False)

    unload_model(model, tokenizer)

print(f"\nВсего экспериментов: {len(all_results)}")

___
## Шаг 4: Сводная таблица результатов

In [ ]:
# Загрузка результатов (если ноутбук перезапущен)
df_prompt = pd.read_csv(RESULTS_DIR / "prompt_results.csv")
df_prompt = df_prompt[~df_prompt["skipped"].astype(bool)]

print("Результаты prompt-based классификации:")
display_cols = [
    "model", "exp_key", "k_shots", "no_desc", "balanced_accuracy",
    "macro_f1", "unknown_rate", "f1_group_A", "f1_group_B", "f1_group_C",
]
df_prompt[display_cols].round(4)

In [ ]:
# Сводная таблица: baseline + prompt
baseline_path = RESULTS_DIR / "classification_results.csv"

if not baseline_path.exists():
    print("Baseline results не найдены — сводная таблица не создаётся.")
else:
    df_baseline = pd.read_csv(baseline_path)

    rows = []
    for _, r in df_baseline.iterrows():
        rows.append({
            "method": "baseline" if r["stage"] == "baseline" else "augmented",
            "model": r["model"],
            "setting": r["stage"],
            "balanced_accuracy": r["balanced_accuracy"],
            "macro_f1": r["macro_f1"],
            "unknown_rate": 0.0,
        })

    for _, r in df_prompt.iterrows():
        rows.append({
            "method": "prompt",
            "model": r["model_name"],
            "setting": f"K={r['k_shots']}_exp{r['exp_key']}",
            "balanced_accuracy": r["balanced_accuracy"],
            "macro_f1": r["macro_f1"],
            "unknown_rate": r["unknown_rate"],
        })

    df_all = pd.DataFrame(rows)
    df_all.to_csv(RESULTS_DIR / "all_methods_comparison.csv", index=False)
    print("Сводная таблица сохранена в results/all_methods_comparison.csv")
    df_all.round(4)

___
## Шаг 5: Визуализации

In [ ]:
# 1. Барплот всех методов по macro_f1
if 'df_all' in dir():
    fig, ax = plt.subplots(figsize=(16, 8))

    df_plot = df_all.copy()
    df_plot["label"] = df_plot["model"] + " (" + df_plot["setting"] + ")"
    df_plot = df_plot.sort_values("macro_f1", ascending=True)

    colors = df_plot["method"].map({"baseline": "#5B9BD5", "augmented": "#70AD47", "prompt": "#FFC000"})

    ax.barh(df_plot["label"], df_plot["macro_f1"], color=colors)
    ax.set_xlabel("Macro F1")
    ax.set_title("Сравнение всех методов по Macro F1")

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#5B9BD5", label="Baseline"),
        Patch(facecolor="#70AD47", label="Augmented"),
        Patch(facecolor="#FFC000", label="Prompt"),
    ]
    ax.legend(handles=legend_elements, loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("df_all не создан — пропуск визуализации")

In [ ]:
# 2. Кривая F1 от K для каждой модели (только эксперименты с описаниями)
fig, ax = plt.subplots(figsize=(10, 6))

for model_key in experiment_matrix:
    df_m = df_prompt[
        (df_prompt["model"] == model_key) &
        (df_prompt["no_desc"] != True)
    ].sort_values("k_shots")
    if len(df_m) > 0:
        ax.plot(df_m["k_shots"], df_m["macro_f1"], marker="o", label=model_key, linewidth=2)

ax.set_xlabel("K (число примеров)")
ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 vs K (с описаниями)")
ax.legend()
ax.set_xticks([0, 1, 3, 5])
plt.tight_layout()
plt.show()

In [ ]:
# 3. Метрики по группам A/B/C
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for idx, group in enumerate(["A", "B", "C"]):
    col = f"f1_group_{group}"
    df_g = df_prompt[["model", "exp_key", "k_shots", col]].dropna()

    for model_key in experiment_matrix:
        df_m = df_g[df_g["model"] == model_key].sort_values("k_shots")
        if len(df_m) > 0:
            axes[idx].plot(df_m["k_shots"], df_m[col], marker="o", label=model_key)

    axes[idx].set_title(f"Группа {group}")
    axes[idx].set_xlabel("K")
    axes[idx].set_xticks([0, 1, 3, 5])
    if idx == 0:
        axes[idx].set_ylabel("F1")
    axes[idx].legend(fontsize=8)

plt.suptitle("F1 по группам классов")
plt.tight_layout()
plt.show()

In [ ]:
# 4. Confusion matrix лучшего метода
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best = df_prompt.loc[df_prompt["macro_f1"].idxmax()]
best_file = RESULTS_DIR / f"preds_{best['model']}_exp{best['exp_key']}.csv"
print(f"Лучший: {best['model']} exp={best['exp_key']} (F1={best['macro_f1']:.4f})")

if best_file.exists():
    df_best = pd.read_csv(best_file)
    df_best = df_best[~df_best["skipped"].astype(bool)]

    cm = confusion_matrix(
        df_best["true_label"], df_best["predicted_label"],
        labels=labels,
    )
    fig, ax = plt.subplots(figsize=(20, 18))
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    disp.plot(ax=ax, xticks_rotation=90, values_format="d", cmap="Blues")
    ax.set_title(f"Confusion Matrix: {best['model']} exp={best['exp_key']}")
    plt.tight_layout()
    plt.show()
else:
    print(f"Файл предсказаний не найден: {best_file}")

In [ ]:
# 5. Per-class F1 с выделением групп цветом
from sklearn.metrics import classification_report

if 'df_best' in dir():
    report = classification_report(
        df_best["true_label"], df_best["predicted_label"],
        labels=labels, output_dict=True, zero_division=0,
    )

    per_class = pd.DataFrame({
        "class": labels,
        "f1": [report[c]["f1-score"] for c in labels],
        "group": [groups.get(c, "?") for c in labels],
    }).sort_values("f1", ascending=True)

    group_colors = {"A": "#5B9BD5", "B": "#70AD47", "C": "#FFC000"}

    fig, ax = plt.subplots(figsize=(14, 10))
    colors = per_class["group"].map(group_colors)
    ax.barh(per_class["class"], per_class["f1"], color=colors)
    ax.set_xlabel("F1-score")
    ax.set_title(f"Per-class F1: {best['model']} exp={best['exp_key']}")

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=c, label=f"Группа {g}")
        for g, c in group_colors.items()
    ]
    ax.legend(handles=legend_elements)
    plt.tight_layout()
    plt.show()
else:
    print("df_best не определён — пропуск")

In [ ]:
# 6. Unknown rate по моделям
fig, ax = plt.subplots(figsize=(10, 6))

for model_key in experiment_matrix:
    df_m = df_prompt[df_prompt["model"] == model_key].sort_values("k_shots")
    if len(df_m) > 0:
        ax.plot(df_m["k_shots"], df_m["unknown_rate"], marker="s", label=model_key, linewidth=2)

ax.set_xlabel("K (число примеров)")
ax.set_ylabel("Unknown rate")
ax.set_title("Доля неизвестных ответов (unknown) по моделям")
ax.legend()
ax.set_xticks([0, 1, 3, 5])
plt.tight_layout()
plt.show()

___
## Итого

Результаты сохранены:
- `results/prompt_results.csv` — все эксперименты prompt-based
- `results/all_methods_comparison.csv` — сводка всех методов
- `results/preds_<model>_exp<key>.csv` — предсказания каждого эксперимента